# 🚀 Kaggle: Huấn luyện Baseline RIFE trên GPU T4 x2 (torchrun DDP siêu tốc)
> Tự động nạp dataset từ RAM trong 2 giây, hỗ trợ Form tương tác chỉnh param giống Colab, đánh giá từng Epoch (Eval Interval = 1) và tự động push kết quả lên GitHub.

In [ ]:
# 1. KÉO CODE TỪ GITHUB VÀ VÀO THƯ MỤC DỰ ÁN
import os
if not os.path.exists('/kaggle/working/RIFE-Project'):
    !git clone https://github.com/NguyenMinhTri24072005/CT282-RIFE.git /kaggle/working/RIFE-Project
%cd /kaggle/working/RIFE-Project/

In [ ]:
#@title 📤 2. CẤU HÌNH GITHUB CREDENTIALS TỪ KAGGLE SECRETS
from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = ""

GITHUB_USERNAME = "NguyenMinhTri24072005" #@param {type:"string"}
GITHUB_EMAIL = "Nguyenminhtri2475n@gmail.com" #@param {type:"string"}
REPO_NAME = "CT282-RIFE" #@param {type:"string"}

GITHUB_TOKEN = token
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "{GITHUB_USERNAME}"
!git remote set-url origin {REPO_URL}

!pip install -q datasets
!git pull origin main

In [ ]:
#@title 🏃 3. Cấu hình và Bắt đầu Huấn luyện Baseline (GPU T4 x2)
ACTIVATION = "prelu" #@param ["prelu", "gelu", "silu", "soft_clamp_relu", "soft_clamp_silu", "smooth_prelu", "optimized_smooth_prelu"]
EPOCHS = 40 #@param {type:"integer"}
BATCH_SIZE = 16 #@param {type:"integer"}
EVAL_INTERVAL = 1 #@param {type:"integer"}
AUTO_PUSH_GITHUB = True #@param {type:"boolean"}

SAVE_DIR = f"trained_model/baseline_{ACTIVATION}"
print(f"📁 Kết quả huấn luyện sẽ được lưu vào: {SAVE_DIR}")

!torchrun --nproc_per_node=2 train.py \
    --model_type original \
    --act {ACTIVATION} \
    --hf_dataset bijinc/vimeo-90k-mini \
    --batch_size {BATCH_SIZE} \
    --epoch {EPOCHS} \
    --eval_interval {EVAL_INTERVAL} \
    --save_dir {SAVE_DIR}

if AUTO_PUSH_GITHUB:
    !git add {SAVE_DIR}/
    !git commit -m "Luu ket qua train baseline_{ACTIVATION} tu Kaggle [Auto Sync]"
    !git push {REPO_URL} main

---
## 🔄 HUẤN LUYỆN TOÀN BỘ CÁC HÀM BASELINE LIÊN TỤC (TÙY CHỌN BATCH)

In [ ]:
#@title ⚡ Chạy Huấn luyện Tự động Toàn bộ các Hàm Baseline (Đánh giá mỗi Epoch)
BATCH_SIZE = 16
EPOCHS = 40
EVAL_INTERVAL = 1
ACTS_TO_TRAIN = ["prelu", "gelu", "silu", "soft_clamp_relu", "soft_clamp_silu", "smooth_prelu"]

for act in ACTS_TO_TRAIN:
    save_path = f"trained_model/baseline_{act}"
    print(f"\n{'='*70}\n🚀 BẮT ĐẦU HUẤN LUYỆN BASELINE: {act.upper()}\n{'='*70}")
    
    !torchrun --nproc_per_node=2 train.py \
        --model_type original \
        --act {act} \
        --hf_dataset bijinc/vimeo-90k-mini \
        --batch_size {BATCH_SIZE} \
        --epoch {EPOCHS} \
        --eval_interval {EVAL_INTERVAL} \
        --save_dir {save_path}
        
    !git add {save_path}/
    !git commit -m "Luu ket qua train baseline_{act} tu Kaggle"
    !git push {REPO_URL} main